## 0. Importación de librerías y paths

In [8]:
from __future__ import annotations
import os
import re
import ast
import time
import json
import hashlib
import subprocess
from pathlib import Path
from typing import Optional

import pandas as pd
import requests


# RUTAS DEL PROYECTO
PROJECT_ROOT = Path(r"C:\Projects\Xeno_Canto_Project")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_RAW = PROJECT_ROOT / "data" / "raw"

AUDIO_CACHE_DIR = DATA_RAW / "audio_cache"
WAV16K_DIR = DATA_INTERIM / "wav16k"

for p in [DATA_PROCESSED, DATA_INTERIM, DATA_RAW, AUDIO_CACHE_DIR, WAV16K_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DF_PATH = DATA_PROCESSED / "df_final.csv"
print("DF_PATH:", DF_PATH)
print("AUDIO_CACHE_DIR:", AUDIO_CACHE_DIR)
print("WAV16K_DIR:", WAV16K_DIR)


DF_PATH: C:\Projects\Xeno_Canto_Project\data\processed\df_final.csv
AUDIO_CACHE_DIR: C:\Projects\Xeno_Canto_Project\data\raw\audio_cache
WAV16K_DIR: C:\Projects\Xeno_Canto_Project\data\interim\wav16k


## 1. Cargar DF y hacer el sanity check de columnas

In [9]:
df = pd.read_csv(DF_PATH)
print("Rows:", len(df), "Cols:", df.shape[1])
print("Columns:", df.columns.tolist())
df.head(3)


Rows: 92797 Cols: 5
Columns: ['scientificName', 'references', 'vernacularName', 'country', 'description']


,scientificName,references,vernacularName,country,description
0,acrocephalus arundinaceus,https://data.biodiversitydata.nl/xeno-canto/ob...,Great Reed Warbler,Sweden,"Acrocephalus arundinaceus, commonly known as t..."
1,acrocephalus arundinaceus,https://data.biodiversitydata.nl/xeno-canto/ob...,Great Reed Warbler,Netherlands,"Acrocephalus arundinaceus, commonly known as t..."
2,acrocephalus arundinaceus,https://data.biodiversitydata.nl/xeno-canto/ob...,Great Reed Warbler,Spain,"Acrocephalus arundinaceus, commonly known as t..."


## 2. Config requests + utilidades (extensión real)

In [10]:
# Unificación de utilidades para descarga y manejo de audio
HEADERS = {
    "User-Agent": "XenoCantoProject/1.0 (contact: anto-rom)",
    "Accept": "*/*",
}

AUDIO_EXT_RE = re.compile(r'(https?://[^\s"\'<>]+?\.(?:mp3|ogg|wav|flac))(?:\?[^\s"\'<>]*)?', re.IGNORECASE)

def looks_like_html(first_bytes: bytes) -> bool:
    head = first_bytes.strip().lower()
    return head.startswith(b"<!doctype html") or head.startswith(b"<html") or b"<title>" in head[:200]

def save_path_for(url: str, out_dir: Path, ext: str) -> Path:
    h = hashlib.sha1(url.encode("utf-8")).hexdigest()[:16]
    return out_dir / f"{h}.{ext}"

def guess_ext_from_content_type(ct: str) -> str:
    ct = (ct or "").lower()
    if "audio/mpeg" in ct or "mpeg" in ct:
        return "mp3"
    if "audio/ogg" in ct or "ogg" in ct:
        return "ogg"
    if "audio/wav" in ct or "wave" in ct:
        return "wav"
    if "audio/flac" in ct or "flac" in ct:
        return "flac"
    return "bin"


## 3. Extraer URLs de audio desde references

In [ ]:
def extract_audio_candidates_from_reference(ref: str, timeout: int = 30) -> list[str]:
    if ref is None or (isinstance(ref, float) and pd.isna(ref)):
        return []

    ref = str(ref).strip()
    if not ref:
        return []

    # 1) Si ya es un audio directo, listo
    if re.search(r"\.(mp3|ogg|wav|flac)(\?|$)", ref, flags=re.IGNORECASE):
        return [ref]

    # 2) Si es una URL a página, intentamos extraer audios del HTML
    if not ref.lower().startswith("http"):
        return []

    try:
        r = requests.get(ref, headers=HEADERS, timeout=timeout, allow_redirects=True)
        if r.status_code >= 400:
            return []

        content = r.content[:2048]
        if not content or not looks_like_html(content):
            # intentamos igualmente regex sobre texto
            pass

        text = r.text  # html
        matches = AUDIO_EXT_RE.findall(text)
        # normalizar y deduplicar manteniendo orden
        out = []
        seen = set()
        for m in matches:
            url = m.strip()
            if url not in seen:
                seen.add(url)
                out.append(url)
        return out

    except Exception:
        return []


## 4. Crear audio_candidates desde la columna references

In [12]:
SOURCE_COL = "references" 

df["audio_candidates"] = df[SOURCE_COL].apply(extract_audio_candidates_from_reference)

# Quick KPI
df["n_candidates"] = df["audio_candidates"].apply(len)
print(df["n_candidates"].describe())
df[["scientificName", "references", "n_candidates"]].head(10)


KeyboardInterrupt: 

## 5. Descarga con fallback

In [ ]:
def download_with_fallback(
    urls: list[str],
    out_dir: Path,
    tries_per_url: int = 3,
    timeout: int = 60,
    min_bytes: int = 20_000
) -> Optional[Path]:
    for url in urls:
        tmp = save_path_for(url, out_dir, "tmp")

        if tmp.exists() and tmp.stat().st_size >= min_bytes:
            return tmp

        for attempt in range(1, tries_per_url + 1):
            try:
                with requests.get(url, headers=HEADERS, stream=True, timeout=timeout, allow_redirects=True) as r:
                    if r.status_code >= 400:
                        raise requests.HTTPError(f"{r.status_code} for {url}", response=r)

                    ct = r.headers.get("Content-Type", "")

                    it = r.iter_content(chunk_size=1024 * 256)
                    first = next(it, b"")
                    if not first or looks_like_html(first):
                        raise RuntimeError(f"Not audio (HTML/empty). Content-Type={ct}")

                    with open(tmp, "wb") as f:
                        f.write(first)
                        for chunk in it:
                            if chunk:
                                f.write(chunk)

                if tmp.stat().st_size < min_bytes:
                    tmp.unlink(missing_ok=True)
                    raise RuntimeError("File too small (likely error page).")

                ext = guess_ext_from_content_type(ct)
                final = save_path_for(url, out_dir, ext if ext != "bin" else "mp3")
                tmp.replace(final)
                return final

            except Exception:
                if attempt < tries_per_url:
                    time.sleep(1.5 * attempt)
                else:
                    tmp.unlink(missing_ok=True)
                    break

    return None


## 6. Conversión a WAW 16kHz mono (requerimiento para Yamnet)

In [ ]:
def to_wav16k_mono(in_path: Path, out_dir: Path) -> Optional[Path]:
    out = out_dir / f"{in_path.stem}_16k.wav"
    if out.exists() and out.stat().st_size > 10_000:
        return out

    cmd = ["ffmpeg", "-y", "-i", str(in_path), "-ac", "1", "-ar", "16000", "-vn", str(out)]
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if out.exists() and out.stat().st_size > 10_000:
            return out
        out.unlink(missing_ok=True)
        return None
    except Exception:
        out.unlink(missing_ok=True)
        return None


## 7. Pipeline por fila (descarga + waw + status)

In [ ]:
def process_row_audio(urls: list[str]) -> tuple[Optional[str], Optional[str], str]:
    if not urls:
        return None, None, "no_candidates"

    p = download_with_fallback(urls, out_dir=AUDIO_CACHE_DIR)
    if p is None:
        return None, None, "download_failed"

    wav = to_wav16k_mono(p, out_dir=WAV16K_DIR)
    if wav is None:
        return str(p), None, "convert_failed"

    return str(p), str(wav), "ok"


## 8. Smoke test (20 filas) y full run

In [ ]:
SAMPLE_N = 20
tmp = df.head(SAMPLE_N).copy()

results = tmp["audio_candidates"].apply(process_row_audio)
tmp[["local_audio_path", "wav16k_path", "audio_status"]] = pd.DataFrame(results.tolist(), index=tmp.index)

print(tmp["audio_status"].value_counts(dropna=False))
tmp[["scientificName", "n_candidates", "audio_status", "wav16k_path"]].head(10)


In [ ]:
results = df["audio_candidates"].apply(process_row_audio)
df[["local_audio_path", "wav16k_path", "audio_status"]] = pd.DataFrame(results.tolist(), index=df.index)

print(df["audio_status"].value_counts(dropna=False).head(20))


## 9. Guardar "train_ready"

In [ ]:
OUT_PATH = DATA_PROCESSED / "df_train_ready.csv"
df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
